# pruning-resnet18 — Training trên Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NhatMinh1503/pruning-resnet18/blob/main/training/colab_training.ipynb)

Notebook dùng chung của nhóm để chạy `training/train.py` (baseline) và `training/run_pruning.py` (cắt + fine-tune).

**Chuẩn bị 1 lần:**
1. Google Drive → *Được chia sẻ với tôi* → chuột phải `pruning-project` → **Sắp xếp → Thêm lối tắt → Drive của tôi**.
2. Colab: **Runtime → Change runtime type → T4 GPU** (notebook này đã đặt sẵn GPU).

| Bước | Thời gian (T4, đo ngày 12/10) |
|---|---|
| Baseline 100 epoch | ~47 phút (~28 giây/epoch) → acc **95.10%** |
| Pruning 2 method × 9 mức × 10 epoch | ~1.5 giờ (ước tính) |

Bị ngắt kết nối thì chạy lại từ ô 1: các script tự học tiếp / bỏ qua phần đã xong.

## 1. Kết nối Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

CKPT = '/content/drive/MyDrive/pruning-project/checkpoints'
RESULTS = '/content/drive/MyDrive/pruning-project/results'
!ls /content/drive/MyDrive/pruning-project

Mounted at /content/drive
benchmark  checkpoints	results  training  研究テーマシート2026後期.docx


## 2. Lấy code mới nhất + kiểm tra GPU

In [2]:
import os
if os.path.isdir('/content/pruning-resnet18'):
    %cd /content/pruning-resnet18
    !git pull
else:
    !git clone https://github.com/NhatMinh1503/pruning-resnet18.git /content/pruning-resnet18
    %cd /content/pruning-resnet18
!pip install -q thop
!nvidia-smi --query-gpu=name --format=csv
# Lấy accuracy.csv chung từ Drive (nếu có) để không ghi đè kết quả cũ
!test -f {RESULTS}/accuracy.csv && cp {RESULTS}/accuracy.csv results/; cat results/accuracy.csv

Cloning into '/content/pruning-resnet18'...
remote: Enumerating objects: 78, done.
remote: Counting objects: 100% (78/78), done.
remote: Compressing objects: 100% (49/49), done.
Receiving objects: 100% (78/78), 34.72 KiB | 17.36 MiB/s, done.
Resolving deltas: 100% (26/26), done.
remote: Total 78 (delta 26), reused 70 (delta 21), pack-reused 0 (from 0)
/content/pruning-resnet18
name
Tesla T4
method,sparsity,acc_before_finetune,acc_after_finetune,seed
baseline,0.0,95.10,95.10,42


## 3A. Train baseline (chỉ chạy nếu chưa có `resnet18_baseline.pt`)

Đã có baseline từ 12/10 (95.10%) → **bỏ qua ô này**.

In [3]:
if os.path.exists(f'{CKPT}/resnet18_baseline.pt'):
    print('Đã có baseline -> bỏ qua')
else:
    !python training/train.py --data-dir /content/data --ckpt-dir {CKPT}

Đã có baseline -> bỏ qua


## 3B. Cắt + fine-tune

Chạy từng method để mỗi lần ~45 phút. Cặp nào đã xong sẽ tự bỏ qua.

In [4]:
!python training/run_pruning.py --data-dir /content/data --ckpt-dir {CKPT} --methods local

100% 170M/170M [28:21<00:00, 100kB/s] 
Baseline accuracy: 95.10%

  local 0.1 ft 1/10 loss=0.0047
  local 0.1 ft 2/10 loss=0.0135
  local 0.1 ft 3/10 loss=0.0196
  local 0.1 ft 4/10 loss=0.0199
  local 0.1 ft 5/10 loss=0.0137
  local 0.1 ft 6/10 loss=0.0085
  local 0.1 ft 7/10 loss=0.0052
  local 0.1 ft 8/10 loss=0.0039
  local 0.1 ft 9/10 loss=0.0032
  local 0.1 ft 10/10 loss=0.0030
[done] local  target=0.1 real=0.100 raw=95.10% finetuned=94.83% (209s)

  local 0.2 ft 1/10 loss=0.0049
  local 0.2 ft 2/10 loss=0.0143
  local 0.2 ft 3/10 loss=0.0197
  local 0.2 ft 4/10 loss=0.0170
  local 0.2 ft 5/10 loss=0.0089
  local 0.2 ft 6/10 loss=0.0076
  local 0.2 ft 7/10 loss=0.0045
  local 0.2 ft 8/10 loss=0.0037
  local 0.2 ft 9/10 loss=0.0030
  local 0.2 ft 10/10 loss=0.0029
[done] local  target=0.2 real=0.200 raw=95.11% finetuned=94.68% (207s)

  local 0.3 ft 1/10 loss=0.0044
  local 0.3 ft 2/10 loss=0.0094
  local 0.3 ft 3/10 loss=0.0135
  local 0.3 ft 4/10 loss=0.0112
  local 0.3 ft 5/10 

In [5]:
!python training/run_pruning.py --data-dir /content/data --ckpt-dir {CKPT} --methods global

Baseline accuracy: 95.10%

  global 0.1 ft 1/10 loss=0.0054
  global 0.1 ft 2/10 loss=0.0171
  global 0.1 ft 3/10 loss=0.0252
  global 0.1 ft 4/10 loss=0.0239
  global 0.1 ft 5/10 loss=0.0157
  global 0.1 ft 6/10 loss=0.0093
  global 0.1 ft 7/10 loss=0.0059
  global 0.1 ft 8/10 loss=0.0041
  global 0.1 ft 9/10 loss=0.0034
  global 0.1 ft 10/10 loss=0.0032
[done] global target=0.1 real=0.100 raw=95.11% finetuned=94.76% (211s)

  global 0.2 ft 1/10 loss=0.0056
  global 0.2 ft 2/10 loss=0.0242
  global 0.2 ft 3/10 loss=0.0265
  global 0.2 ft 4/10 loss=0.0207
  global 0.2 ft 5/10 loss=0.0132
  global 0.2 ft 6/10 loss=0.0077
  global 0.2 ft 7/10 loss=0.0051
  global 0.2 ft 8/10 loss=0.0043
  global 0.2 ft 9/10 loss=0.0032
  global 0.2 ft 10/10 loss=0.0029
[done] global target=0.2 real=0.200 raw=95.12% finetuned=94.88% (209s)

  global 0.3 ft 1/10 loss=0.0068
  global 0.3 ft 2/10 loss=0.0203
  global 0.3 ft 3/10 loss=0.0197
  global 0.3 ft 4/10 loss=0.0155
  global 0.3 ft 5/10 loss=0.0117
  

## 4. Lưu kết quả lên Drive chung

In [6]:
!cp results/accuracy.csv {RESULTS}/
!cat {RESULTS}/accuracy.csv
!ls -lh {CKPT}

method,sparsity,acc_before_finetune,acc_after_finetune,seed
baseline,0.0,95.10,95.10,42
local,0.1,95.10,94.83,42
local,0.2,95.11,94.68,42
local,0.3,95.02,94.84,42
local,0.4,94.89,94.95,42
local,0.5,94.40,94.90,42
local,0.6,93.69,94.92,42
local,0.7,90.19,94.85,42
local,0.8,63.42,94.77,42
local,0.9,24.49,94.10,42
global,0.1,95.11,94.76,42
global,0.2,95.12,94.88,42
global,0.3,95.10,94.75,42
global,0.4,95.09,95.01,42
global,0.5,94.94,94.90,42
global,0.6,94.66,94.90,42
global,0.7,93.99,95.11,42
global,0.8,88.55,94.89,42
global,0.9,57.13,94.69,42
total 1.6G
-rw------- 1 root root 43M Oct 12 08:36 resnet18_baseline.pt
-rw------- 1 root root 43M Oct 15 01:49 resnet18_global_0.1_finetuned.pt
-rw------- 1 root root 43M Oct 15 01:46 resnet18_global_0.1_raw.pt
-rw------- 1 root root 43M Oct 15 01:53 resnet18_global_0.2_finetuned.pt
-rw------- 1 root root 43M Oct 15 01:49 resnet18_global_0.2_raw.pt
-rw------- 1 root root 43M Oct 15 01:56 resnet18_global_0.3_finetuned.pt
-rw------- 1 root root 43M O